# Chapter 4 &mdash; Visitation Numbers, Pumping Up and Down

**Concept 13 of the Chapter 4 decomposition:** *Visitation Numbers, Pumping Up and Pumping Down*

Emboss a number at each visit. A twice-embossed state carries a pump you can skip or repeat.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Pumping-Up-And-Down/Concept-Pumping-Up-And-Down.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Imagine a rubber stamp **embossing a visitation number** on each state as you arrive.
Running a string of length $M \ge N$ through an $N$-state DFA, some state $s_p$ gets
stamped twice &mdash; at $v_p$ and $v_{p+k}$. **Any state embossed twice carries a pump**
of length $k>0$.

Two further journeys must then exist:

* **pumping down** &mdash; skip the loop;
* **pumping up** &mdash; take it again, and again.

The DFA is **forced** to admit all of them.

## 2. Definitions

### Emboss visitation numbers

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')

def emboss(D, s):
    q, stamps = D["q0"], [(0, D["q0"])]
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        stamps.append((i, q))
    return stamps

### Locate the first pump

In [ ]:
def first_pump(D, s):
    seen = {}
    q = D["q0"]; seen[q] = 0
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        if q in seen:
            return seen[q], i, q            # v_p, v_{p+k}, state
        seen[q] = i
    return None

## 3. Tests

The stamps, and the first repeat.

In [ ]:
w = '0100'
for i, q in emboss(L3Z, w): print("  v%-2d -> %s" % (i, q))
p, pk, st = first_pump(L3Z, w)
print("\nfirst pump: state %s stamped at v%d and v%d, so k = %d" % (st, p, pk, pk-p))
assert pk > p

Split accordingly, then pump **down** and **up**.

In [ ]:
p, pk, st = first_pump(L3Z, w)
x, y, z = w[:p], w[p:pk], w[pk:]
print("x=%r  y=%r  z=%r   (y non-empty: %s, |xy| = %d)" % (x, y, z, y != '', len(x+y)))
assert y != ''
print("\npump down (i=0):", repr(x + z),      "accepted?", accepts_dfa(L3Z, x + z))
for i in [1,2,3,5]:
    s = x + y*i + z
    print("pump up   (i=%d): %-14r accepted? %s" % (i, s, accepts_dfa(L3Z, s)))
assert all(accepts_dfa(L3Z, x + y*i + z) for i in range(8))

The machine cannot distinguish them, so it must accept them all.

In [ ]:
ends = {run_dfa(L3Z, x + y*i + z) for i in range(8)}
print("final state for every i :", ends)
assert len(ends) == 1, "all pumped strings end in the SAME state"

## 4. Exercises


1. Why do we focus on the **first** pump? What would change if we chose another?
2. Find a string with **two** distinct pumps. Does pumping either one work?
3. What is the largest $|xy|$ can be, and why does that matter later?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter4/Concept-Pumping-Up-And-Down')